# Portable Artifact Export (v3)

Owner: Jessica Bedson. Follow-up to Task 1.6, requested by Mark (Nov 2026).

**Problem.** The frozen v3 models were saved with pickle under scikit-learn 1.6.1. A pickle stores scikit-learn's private internals, so it fails under other versions (Mark's environment, scikit-learn 1.9.1, raises on the first transform). `score_firm` therefore cannot run outside the build environment.

**Fix.** Save the same model in a form that does not depend on any library version:

| Component | Saved as |
|---|---|
| Feature order, prior-opinion column, model version, default probability, clip bounds | plain JSON |
| Median imputer | fill value per feature, kept columns, missing-indicator columns (plain JSON) |
| Calibrator (logistic regression on log-odds) | coefficient and intercept (plain JSON) |
| XGBoost booster | `Booster.save_model` JSON, XGBoost's documented, version-stable format |

Files written next to the originals in `models/`: `<version>.portable.json` and `<version>.booster.json`. The `.pkl` files are **not** changed.

**Check.** Every assessed firm-year in the panel is scored with the original pickle and with the portable artifact, and the probabilities must match to four decimal places on every row. The notebook stops if they do not. The row-level comparison is saved to `results/portable_export_check_v3.csv`.

**Run this in the environment that built the models** (Python 3.13.5, scikit-learn 1.6.1, xgboost 3.2.0), because reading the pickle needs that environment. The new `scoring.py` must be in the same folder.

## Setup

In [1]:
import hashlib
import json
import pickle
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import xgboost

import scoring
assert hasattr(scoring, "load_artifact"), "This folder has the old scoring.py. Use the new one (with load_artifact)."

DATA_PATH = Path("final_panel 1.csv")
MODEL_DIR = Path("models")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

decisions = json.loads((MODEL_DIR / "decisions_v3.json").read_text(encoding="utf-8"))
VERSIONS = [decisions["primary_version"], decisions["variant_version"]]

EXPORT_ENV = {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
              "scikit_learn": sklearn.__version__, "xgboost": xgboost.__version__}
print("Exporting:", VERSIONS)
print("This environment:", EXPORT_ENV)

Exporting: ['financial_only_xgb_v3', 'prior_opinion_xgb_v3']
This environment: {'python': '3.13.5', 'pandas': '2.2.3', 'numpy': '2.1.3', 'scikit_learn': '1.6.1', 'xgboost': '3.2.0'}


## Export function

Reads the fitted objects' state and writes it as plain data. The imputer reproduces scikit-learn's `SimpleImputer(strategy="median", add_indicator=True)`: missing values are replaced by the training median, columns that were entirely missing in training are dropped, and one 0/1 indicator is appended for each column that had missing values in training, in that column order. Supported pipelines: imputer, optional standard scaler, then XGBoost or logistic regression. Anything else stops with an error rather than being exported approximately.

In [2]:
PORTABLE_FORMAT = "portable-model-v1"


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def export_portable(version_id):
    pkl_path = MODEL_DIR / f"{version_id}.pkl"
    art = pickle.loads(pkl_path.read_bytes())
    pipe = art["pipeline"]
    steps = dict(pipe.named_steps)
    imp, clf = steps["imputer"], steps["classifier"]
    if set(steps) - {"imputer", "scaler", "classifier"}:
        raise ValueError(f"Unexpected pipeline steps: {list(steps)}")

    # Imputer
    if imp.strategy != "median" or not (isinstance(imp.missing_values, float) and np.isnan(imp.missing_values)):
        raise ValueError("Only a median imputer with missing_values=np.nan is supported.")
    if getattr(imp, "keep_empty_features", False):
        raise ValueError("keep_empty_features=True is not supported.")
    stats = np.asarray(imp.statistics_, dtype=float)
    if imp.n_features_in_ != len(art["feature_cols"]):
        raise ValueError("Imputer input width does not match feature_cols.")
    imputer = {
        "strategy": "median",
        "input_columns": list(art["feature_cols"]),
        "fill_values": [None if np.isnan(v) else float(v) for v in stats],
        "kept_columns": [int(i) for i in np.flatnonzero(~np.isnan(stats))],
        "indicator_columns": [int(i) for i in imp.indicator_.features_] if imp.add_indicator else [],
    }

    # Optional scaler
    scaler = None
    if "scaler" in steps:
        sc = steps["scaler"]
        scaler = {"mean": [float(v) for v in sc.mean_], "scale": [float(v) for v in sc.scale_]}

    # Classifier
    if isinstance(clf, xgboost.XGBClassifier):
        booster_file = f"{version_id}.booster.json"
        clf.get_booster().save_model(str(MODEL_DIR / booster_file))
        classifier = {"type": "xgboost", "booster_file": booster_file,
                      "booster_format": "XGBoost JSON model (Booster.save_model)",
                      "booster_sha256": sha256(MODEL_DIR / booster_file)}
    elif clf.__class__.__name__ == "LogisticRegression":
        classifier = {"type": "logistic", "coef": [float(v) for v in clf.coef_.ravel()],
                      "intercept": float(clf.intercept_[0])}
    else:
        raise ValueError(f"{clf.__class__.__name__} has no version-stable export; not supported.")

    # Calibrator: logistic regression with one input (the log-odds)
    cal = art["calibrator"]
    if cal.coef_.shape != (1, 1):
        raise ValueError("Calibrator must have a single coefficient.")

    spec = {
        "format": PORTABLE_FORMAT,
        "model_version": art["model_version"],
        "model_name": art["model_name"],
        "feature_cols": list(art["feature_cols"]),
        "prior_opinion_col": art.get("prior_opinion_col"),
        "default_probability": art["default_probability"],
        "calibration_transform": art["calibration_transform"],
        "target_definition": art["target_definition"],
        "imputer": imputer,
        "scaler": scaler,
        "classifier": classifier,
        "calibrator": {"type": "logistic_on_log_odds", "coef": float(cal.coef_[0, 0]),
                       "intercept": float(cal.intercept_[0])},
        "source_pickle": pkl_path.name,
        "source_pickle_sha256": sha256(pkl_path),
        "data_source_sha256": art.get("source_sha256"),
        "random_seed": art.get("random_seed"),
        "frozen_at_utc": art.get("frozen_at_utc"),
        "built_under": art.get("versions"),
        "exported_under": EXPORT_ENV,
        "exported_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    out = MODEL_DIR / f"{version_id}.portable.json"
    out.write_text(json.dumps(spec, indent=2, allow_nan=False), encoding="utf-8")
    return art, out

## Export both frozen models

In [3]:
originals, portables = {}, {}
for version_id in VERSIONS:
    art, out = export_portable(version_id)
    originals[version_id] = art
    portables[version_id] = scoring.load_artifact(out)
    files = [out.name] + ([portables[version_id]["classifier"]["booster_file"]]
                          if portables[version_id]["classifier"]["type"] == "xgboost" else [])
    print(f"{version_id}: wrote {', '.join(files)}  (original {version_id}.pkl unchanged)")

financial_only_xgb_v3: wrote financial_only_xgb_v3.portable.json, financial_only_xgb_v3.booster.json  (original financial_only_xgb_v3.pkl unchanged)
prior_opinion_xgb_v3: wrote prior_opinion_xgb_v3.portable.json, prior_opinion_xgb_v3.booster.json  (original prior_opinion_xgb_v3.pkl unchanged)


## Comparison set: every assessed firm-year in the panel

These are all the firm-years `score_firm` can be asked to score (auopic 1 or 2, FY2011 to FY2022), prepared exactly as `score_firm_core` prepares them.

In [4]:
raw_panel = pd.read_csv(DATA_PATH, dtype={"gvkey": "string", "cik": "string", "cusip": "string", "tic": "string"},
                        parse_dates=["datadate", "INFO_DATE"])
panel = scoring._prepare(raw_panel.loc[raw_panel["auopic"].isin([1, 2])].copy())
panel["prior_adverse_opinion"] = (panel["auopic"] == 2).astype(int)
panel = panel.sort_values(["fyear", "gvkey"]).reset_index(drop=True)
print(f"Firm-years compared: {len(panel):,} across {panel.gvkey.nunique():,} firms, FY{panel.fyear.min()}-{panel.fyear.max()}")

Firm-years compared: 42,728 across 6,187 firms, FY2011-2022


## Original pickle versus portable artifact

The original path is the one that produced the reported results: the pickled pipeline's `predict_proba`, the log-odds in float64, then the pickled calibrator. The portable path is `scoring.predict_raw` and `scoring.calibrate`. The preprocessing output is compared as well, so a mismatch would show where it starts.

In [5]:
def log_odds(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p)).reshape(-1, 1)

rows, summary = [], {}
for version_id in VERSIONS:
    old, new = originals[version_id], portables[version_id]
    X = panel[old["feature_cols"]]

    pre_old = old["pipeline"][:-1].transform(X)
    pre_new = scoring.transform(new, X)
    assert pre_old.shape == pre_new.shape, f"{version_id}: preprocessing width differs."
    pre_diff = float(np.max(np.abs(pre_old - pre_new)))

    raw_old = old["pipeline"].predict_proba(X)[:, 1].astype(float)
    cal_old = old["calibrator"].predict_proba(log_odds(raw_old))[:, 1]
    raw_new = scoring.predict_raw(new, X)
    cal_new = scoring.calibrate(new, raw_new)

    diff = np.abs(cal_old - cal_new)
    match4 = np.round(cal_old, 4) == np.round(cal_new, 4)
    summary[version_id] = {
        "rows": int(len(X)),
        "preprocessing_max_abs_diff": pre_diff,
        "raw_max_abs_diff": float(np.max(np.abs(raw_old - raw_new))),
        "calibrated_max_abs_diff": float(diff.max()),
        "rows_matching_4dp": int(match4.sum()),
        "rows_not_matching_4dp": int((~match4).sum()),
    }
    rows.append(pd.DataFrame({
        "model_version": version_id, "gvkey": panel["gvkey"].astype(str), "fyear": panel["fyear"].astype(int),
        "p_pickle": cal_old, "p_portable": cal_new, "abs_diff": diff, "match_4dp": match4,
    }))

check = pd.concat(rows, ignore_index=True)
check.to_csv(RESULTS_DIR / "portable_export_check_v3.csv", index=False)
print(pd.DataFrame(summary).T.to_string())
for version_id, s in summary.items():
    assert s["rows_not_matching_4dp"] == 0, f"{version_id}: {s['rows_not_matching_4dp']} rows differ at 4 dp. Do not use."
print(f"\nAll {len(check):,} comparisons match to four decimal places. Row-level file: results/portable_export_check_v3.csv")

                          rows  preprocessing_max_abs_diff  raw_max_abs_diff  calibrated_max_abs_diff  rows_matching_4dp  rows_not_matching_4dp
financial_only_xgb_v3  42728.0                         0.0               0.0                      0.0            42728.0                    0.0
prior_opinion_xgb_v3   42728.0                         0.0               0.0                      0.0            42728.0                    0.0

All 85,456 comparisons match to four decimal places. Row-level file: results/portable_export_check_v3.csv


## Cross-check against the reported test results

If `results/test_predictions_v3.csv` is present, the portable artifact must reproduce the test-set probabilities that the reported metrics were computed from.

In [6]:
pred_path = RESULTS_DIR / "test_predictions_v3.csv"
if pred_path.exists():
    reported = pd.read_csv(pred_path, dtype={"gvkey": "string"})
    merged = reported.merge(
        check.pivot_table(index=["gvkey", "fyear"], columns="model_version", values="p_portable").reset_index(),
        left_on=["gvkey", "feature_year"], right_on=["gvkey", "fyear"], how="left", validate="one_to_one",
    )
    assert merged[VERSIONS].notna().all().all(), "Some test firm-years were not scored."
    d_fin = float(np.max(np.abs(merged["p_financial"] - merged[VERSIONS[0]])))
    d_pri = float(np.max(np.abs(merged["p_prior"] - merged[VERSIONS[1]])))
    print(f"Test predictions ({len(merged):,} rows): max abs difference {d_fin:.2e} (financial-only), {d_pri:.2e} (prior-opinion)")
    assert max(d_fin, d_pri) < 5e-5
    summary["test_predictions_cross_check"] = {"rows": int(len(merged)), "max_abs_diff_financial": d_fin,
                                                "max_abs_diff_prior": d_pri}
else:
    print("results/test_predictions_v3.csv not found; cross-check skipped.")

Test predictions (12,325 rows): max abs difference 9.97e-17 (financial-only), 9.97e-17 (prior-opinion)


## End-to-end check through score_firm_core

Scores a random sample of firm-years through the updated `score_firm_core`, which loads the portable artifact, and compares each result with the original pickle's probability.

In [7]:
rng = np.random.default_rng(20260928)
sample = panel.iloc[rng.choice(len(panel), size=200, replace=False)]
e2e = []
for version_id in VERSIONS:
    ref = check[check.model_version == version_id].set_index(["gvkey", "fyear"])["p_pickle"]
    for _, row in sample.iterrows():
        res = scoring.score_firm_core(row["gvkey"], row["fyear"], raw_panel, portables[version_id])
        e2e.append(abs(res["probability"] - ref.loc[(str(row["gvkey"]), int(row["fyear"]))]))
e2e = np.array(e2e)
assert e2e.max() < 5e-5
summary["score_firm_core_end_to_end"] = {"calls": int(len(e2e)), "max_abs_diff": float(e2e.max())}
print(f"score_firm_core on {len(e2e)} calls (200 firm-years x 2 models): max abs difference {e2e.max():.2e}")

score_firm_core on 400 calls (200 firm-years x 2 models): max abs difference 0.00e+00


## Environment record (for the model card)

In [8]:
built = originals[VERSIONS[0]]["versions"]
summary["built_under"] = built
summary["exported_under"] = EXPORT_ENV
(RESULTS_DIR / "portable_export_check_v3_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Original pickles were built under:", json.dumps(built))
print("Portable files were exported under:", json.dumps(EXPORT_ENV))
print("\nSummary written to results/portable_export_check_v3_summary.json")

Original pickles were built under: {"python": "3.13.5", "pandas": "2.2.3", "numpy": "2.1.3", "scikit_learn": "1.6.1", "xgboost": "3.2.0"}
Portable files were exported under: {"python": "3.13.5", "pandas": "2.2.3", "numpy": "2.1.3", "scikit_learn": "1.6.1", "xgboost": "3.2.0"}

Summary written to results/portable_export_check_v3_summary.json
